# Option 4 - URL Analysis

## Imports

In [1]:
from urllib.parse import urlparse, parse_qs
import requests

## Input URLs

In [2]:
urls = [
    "https://edition.cnn.com/2025/12/22/media/60-minutes-cecot-bari-weiss-canada-global-tv?iid=cnn_buildContentRecirc_end_recirc&recs_exp=most-popular-article-end&tenant_id=popular.en",
    "https://www.nhm.ac.uk/visit/exhibitions/wildlife-photographer-of-the-year.html",
    "https://is-web.hevra.haifa.ac.il/images/2025_SEM._aa.pdf"
]

## Part A - Extract URL Components

For each URL we extract: TLD, domain, subdomain, path, file name, query key-value pairs, and port.

In [3]:
def get_tld_domain_subdomain(hostname):
    """
    Split hostname into subdomain, domain, TLD.
    Handles two-part TLDs like ac.uk, ac.il, co.uk.
    """
    two_part_tlds = ["ac.uk", "co.uk", "ac.il", "co.il", "gov.uk", "org.uk"]

    for tld in two_part_tlds:
        if hostname.endswith("." + tld):
            remainder = hostname[: -(len(tld) + 1)]  # strip .tld
            parts = remainder.split(".")
            domain = parts[-1]
            subdomain = ".".join(parts[:-1])
            return subdomain, domain, tld

    # Regular single-part TLD (e.g. .com)
    parts = hostname.split(".")
    tld = parts[-1]
    domain = parts[-2] if len(parts) >= 2 else ""
    subdomain = ".".join(parts[:-2]) if len(parts) > 2 else ""
    return subdomain, domain, tld


def extract_url_components(url):
    parsed = urlparse(url)

    hostname = parsed.hostname
    subdomain, domain, tld = get_tld_domain_subdomain(hostname)

    # Port: use explicit port or default based on scheme
    if parsed.port:
        port = parsed.port
    elif parsed.scheme == "https":
        port = 443
    else:
        port = 80

    # File name: last segment of path if it has an extension
    path_segments = parsed.path.rstrip("/").split("/")
    last_segment = path_segments[-1] if path_segments else ""
    filename = last_segment if "." in last_segment else ""

    # Query key-value pairs
    query_params = parse_qs(parsed.query)
    # flatten single-value lists
    query_params = {k: v[0] if len(v) == 1 else v for k, v in query_params.items()}

    return {
        "tld": tld,
        "domain": domain,
        "subdomain": subdomain,
        "path": parsed.path,
        "filename": filename,
        "query_params": query_params,
        "port": port,
        "hostname": hostname,
        "scheme": parsed.scheme
    }


for url in urls:
    info = extract_url_components(url)
    print("URL:", url)
    print("  TLD:        ", info["tld"])
    print("  Domain:     ", info["domain"])
    print("  Subdomain:  ", info["subdomain"] if info["subdomain"] else "(none)")
    print("  Path:       ", info["path"])
    print("  Filename:   ", info["filename"] if info["filename"] else "(none)")
    print("  Query params:", info["query_params"] if info["query_params"] else "(none)")
    print("  Port:       ", info["port"])
    print()

URL: https://edition.cnn.com/2025/12/22/media/60-minutes-cecot-bari-weiss-canada-global-tv?iid=cnn_buildContentRecirc_end_recirc&recs_exp=most-popular-article-end&tenant_id=popular.en
  TLD:         com
  Domain:      cnn
  Subdomain:   edition
  Path:        /2025/12/22/media/60-minutes-cecot-bari-weiss-canada-global-tv
  Filename:    (none)
  Query params: {'iid': 'cnn_buildContentRecirc_end_recirc', 'recs_exp': 'most-popular-article-end', 'tenant_id': 'popular.en'}
  Port:        443

URL: https://www.nhm.ac.uk/visit/exhibitions/wildlife-photographer-of-the-year.html
  TLD:         ac.uk
  Domain:      nhm
  Subdomain:   www
  Path:        /visit/exhibitions/wildlife-photographer-of-the-year.html
  Filename:    wildlife-photographer-of-the-year.html
  Query params: (none)
  Port:        443

URL: https://is-web.hevra.haifa.ac.il/images/2025_SEM._aa.pdf
  TLD:         ac.il
  Domain:      haifa
  Subdomain:   is-web.hevra
  Path:        /images/2025_SEM._aa.pdf
  Filename:    2025_SE

## Part B - Check robots.txt

For each URL we look for a `robots.txt` file under the subdomain (base URL).

In [4]:
def get_robots_txt(base_url):
    robots_url = base_url + "/robots.txt"
    try:
        response = requests.get(robots_url, timeout=5)
        if response.status_code == 200:
            return response.text
        else:
            return None
    except:
        return None


def parse_robots_txt(text):
    disallowed = []   # (user_agent, path)
    user_agents = []
    crawl_delay = None
    current_agents = []

    for line in text.splitlines():
        line = line.split("#")[0].strip()  # remove comments
        if not line:
            current_agents = []
            continue
        if line.lower().startswith("user-agent:"):
            agent = line.split(":", 1)[1].strip()
            current_agents.append(agent)
            if agent not in user_agents:
                user_agents.append(agent)
        elif line.lower().startswith("disallow:"):
            path = line.split(":", 1)[1].strip()
            if path:
                for agent in current_agents:
                    disallowed.append((agent, path))
        elif line.lower().startswith("crawl-delay:"):
            crawl_delay = line.split(":", 1)[1].strip()

    return disallowed, user_agents, crawl_delay


for url in urls:
    info = extract_url_components(url)
    base_url = info["scheme"] + "://" + info["hostname"]

    print("URL:", url)
    print("Checking robots.txt at:", base_url + "/robots.txt")

    robots_text = get_robots_txt(base_url)

    if robots_text is None:
        print("  No robots.txt found.")
    else:
        print("  robots.txt found!")
        disallowed, user_agents, crawl_delay = parse_robots_txt(robots_text)

        # B.1 - Disallowed addresses
        print("\n  B.1 - Disallowed addresses:")
        if disallowed:
            for agent, path in disallowed:
                print(f"    [{agent}] {base_url}{path}")
        else:
            print("    (none)")

        # B.2 - Prohibited User-agents
        print("\n  B.2 - User-agents listed:")
        for agent in user_agents:
            print("   ", agent)

        # B.3 - Crawl delay
        print("\n  B.3 - Crawl-delay:", crawl_delay if crawl_delay else "(none specified)")

    print()

URL: https://edition.cnn.com/2025/12/22/media/60-minutes-cecot-bari-weiss-canada-global-tv?iid=cnn_buildContentRecirc_end_recirc&recs_exp=most-popular-article-end&tenant_id=popular.en
Checking robots.txt at: https://edition.cnn.com/robots.txt
  robots.txt found!

  B.1 - Disallowed addresses:
    [AI2Bot] https://edition.cnn.com/
    [Ai2Bot-Dolma] https://edition.cnn.com/
    [Amazonbot] https://edition.cnn.com/
    [amzn-searchbot] https://edition.cnn.com/
    [amzn-user] https://edition.cnn.com/
    [anthropic-ai] https://edition.cnn.com/
    [Applebot-Extended] https://edition.cnn.com/
    [AwarioRssBot] https://edition.cnn.com/
    [AwarioSmartBot] https://edition.cnn.com/
    [Brightbot 1.0] https://edition.cnn.com/
    [Bytespider] https://edition.cnn.com/
    [CCBot] https://edition.cnn.com/
    [ChatGPT-User] https://edition.cnn.com/
    [ClaudeBot] https://edition.cnn.com/
    [Claude-SearchBot] https://edition.cnn.com/
    [Claude-Web] https://edition.cnn.com/
    [cohere-ai

## Summary

Collecting all information gathered for each URL in one place.

In [5]:
for i, url in enumerate(urls, 1):
    info = extract_url_components(url)
    base_url = info["scheme"] + "://" + info["hostname"]
    robots_text = get_robots_txt(base_url)

    print(f"======= URL {i} =======")
    print(f"Full URL:     {url}")
    print()
    print("-- Part A: URL Components --")
    print(f"  TLD:          {info['tld']}")
    print(f"  Domain:       {info['domain']}")
    print(f"  Subdomain:    {info['subdomain'] if info['subdomain'] else '(none)'}")
    print(f"  Path:         {info['path']}")
    print(f"  Filename:     {info['filename'] if info['filename'] else '(none)'}")
    print(f"  Query params: {info['query_params'] if info['query_params'] else '(none)'}")
    print(f"  Port:         {info['port']}")
    print()
    print("-- Part B: robots.txt --")
    if robots_text is None:
        print("  No robots.txt found.")
    else:
        disallowed, user_agents, crawl_delay = parse_robots_txt(robots_text)
        print(f"  User-agents:  {user_agents}")
        print(f"  Disallowed:   {[base_url + p for _, p in disallowed] if disallowed else '(none)'}")
        print(f"  Crawl-delay:  {crawl_delay if crawl_delay else '(none)'}")
    print()


======= URL 1 =======
Full URL:     https://edition.cnn.com/2025/12/22/media/60-minutes-cecot-bari-weiss-canada-global-tv?iid=cnn_buildContentRecirc_end_recirc&recs_exp=most-popular-article-end&tenant_id=popular.en

-- Part A: URL Components --
  TLD:          com
  Domain:       cnn
  Subdomain:    edition
  Path:         /2025/12/22/media/60-minutes-cecot-bari-weiss-canada-global-tv
  Filename:     (none)
  Query params: {'iid': 'cnn_buildContentRecirc_end_recirc', 'recs_exp': 'most-popular-article-end', 'tenant_id': 'popular.en'}
  Port:         443

-- Part B: robots.txt --
  User-agents:  ['AI2Bot', 'Ai2Bot-Dolma', 'Amazonbot', 'amzn-searchbot', 'amzn-user', 'anthropic-ai', 'Applebot-Extended', 'AwarioRssBot', 'AwarioSmartBot', 'Brightbot 1.0', 'Bytespider', 'CCBot', 'ChatGPT-User', 'ClaudeBot', 'Claude-SearchBot', 'Claude-Web', 'cohere-ai', 'cohere-training-data-crawler', 'Crawlspace', 'DataForSeoBot', 'Diffbot', 'DuckAssistBot', 'EchoboxBot', 'FacebookBot', 'FriendlyCrawler', 'G